In [ ]:

!pip install geopandas frechetdist shapely rtree scipy joblib

import geopandas as gpd
import numpy as np
from joblib import Parallel, delayed
from frechetdist import frdist
from scipy.sparse import lil_matrix, save_npz
import pandas as pd
import os



gdf = gpd.read_file("../../data/processed/segments.gpkg") 

gdf = gdf.reset_index(drop=True)


clean_trajectories = []
valid_indices = []

for idx, geom in enumerate(gdf.geometry):

    if geom is None or geom.is_empty:
        continue

    coords = list(geom.coords)

    if len(coords) < 2:
        continue

    try:
        arr = np.array([[float(c[0]), float(c[1])] for c in coords], dtype=np.float64)

        if not np.isfinite(arr).all():
            continue

        if arr.ndim != 2 or arr.shape[1] != 2:
            continue

        clean_trajectories.append(arr)
        valid_indices.append(idx)

    except:
        continue

gdf = gdf.iloc[valid_indices].reset_index(drop=True)
trajectories = clean_trajectories



In [ ]:
def get_candidate_pairs(gdf, buffer_dist=0.01):
    pairs = set()
    sindex = gdf.sindex

    for i in range(len(gdf)):
        geom = gdf.geometry.iloc[i]
        bounds = geom.buffer(buffer_dist).bounds

        candidates = list(sindex.intersection(bounds))

        for j in candidates:
            if i < j:
                pairs.add((i, j))

    return list(pairs)


pairs = get_candidate_pairs(gdf, buffer_dist=0.01)
print(f"Candidate pairs: {len(pairs)}")

In [ ]:
def chunked_pairs(pairs, chunk_size=100):
    for i in range(0, len(pairs), chunk_size):
        yield pairs[i:i + chunk_size]


def compute_chunk(pair_chunk):
    out = []

    for i, j in pair_chunk:
        t1 = trajectories[i]
        t2 = trajectories[j]

        try:
            if t1.shape[1] != 2 or t2.shape[1] != 2:
                continue
            if not np.isfinite(t1).all() or not np.isfinite(t2).all():
                continue

            d = frdist(t1, t2)
            out.append((i, j, d))

        except:
            continue

    return out

In [ ]:


num_workers = os.cpu_count()

results_nested = Parallel(n_jobs=num_workers, verbose=10)(
    delayed(compute_chunk)(chunk)
    for chunk in chunked_pairs(pairs, chunk_size=100)
)

# flatten
results = [item for sublist in results_nested for item in sublist]


In [ ]:
N = len(trajectories)
dist_matrix = lil_matrix((N, N), dtype=np.float32)

for i, j, d in results:
    dist_matrix[i, j] = d
    dist_matrix[j, i] = d

dist_matrix = dist_matrix.tocsr()


save_npz("frechet_matrix_segments_cleaned.npz", dist_matrix)

df_results = pd.DataFrame(results, columns=["i", "j", "distance"])
df_results.to_csv("frechet_pairs_segments_cleaned.csv", index=False)


In [ ]:
import pandas as pd

df = pd.DataFrame(results, columns=["i", "j", "distance"])
df.to_parquet("../../data/processed/frechet_results.parquet")